# Dataset Characterization: TMDB

Initial characterization EDA for the TMDB dataset used in this
dissertation: the Full TMDB Movies Dataset 2024. Findings are logged to
`reports/eda_log.md` via `src/eda/report.py`; figures are saved to
`reports/figures/` at 300dpi.

## Setup

### Imports & configuration

In [ ]:
%matplotlib inline

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda import config
from src.eda.helper import (
    empty_string_report,
    entity_embedding_dim,
    flag_non_english,
    high_corr_pairs,
    html_contamination_pct,
    iqr_outlier_count,
    list_field_summary,
    multihot_feasibility,
    near_duplicate_report,
    parse_list_field,
    text_field_report,
    token_length_stats,
    top_item_cooccurrence,
    zscore_outlier_count,
)
from src.eda.loaders import load_tmdb
from src.eda.report import log_findings, savefig

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

FIGURES_DIR = REPO_ROOT / "reports" / "figures"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import sys
print(sys.executable)
import torch
print("torch imported fine")


### Load datasets

In [ ]:
tmdb_raw = load_tmdb()

print("TMDB raw shape:", tmdb_raw.shape)

## 1. Initial checks

### 1.1 Provenance & structural overview

#### Dataset provenance

In [ ]:
display(Markdown(f"""
**Dataset provenance** (pulled from `src/eda/config.py`):

- TMDB download date: `{config.TMDB_DOWNLOAD_DATE}`
"""))

#### Shape, dtypes & column list

In [ ]:
tmdb_shape_raw = tmdb_raw.shape
print("TMDB shape:", tmdb_shape_raw)

In [ ]:
tmdb_raw.dtypes

In [ ]:
print("TMDB columns:")
print(list(tmdb_raw.columns))

#### Memory usage

In [ ]:
tmdb_mem_mb = tmdb_raw.memory_usage(deep=True).sum() / 1e6
print(f"TMDB memory usage (deep): {tmdb_mem_mb:,.1f} MB")

CHUNKED_LOAD_THRESHOLD_MB = 2000
if tmdb_mem_mb > CHUNKED_LOAD_THRESHOLD_MB:
    print(f"FLAG: TMDB exceeds {CHUNKED_LOAD_THRESHOLD_MB} MB — consider chunked loading.")
else:
    print(f"TMDB is within comfortable in-memory limits ({tmdb_mem_mb:,.1f} MB).")

#### Manual sample inspection

In [ ]:
tmdb_raw.sample(10, random_state=42)

#### Missingness

In [ ]:
tmdb_missing = pd.DataFrame(
    {
        "n_missing": tmdb_raw.isnull().sum(),
        "pct_missing": tmdb_raw.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
tmdb_missing

In [ ]:
msno.matrix(tmdb_raw.sample(min(len(tmdb_raw), 5000), random_state=42))
plt.title("TMDB missingness pattern (sample)")
fig = plt.gcf()
savefig(fig, "00_tmdb_missingness_matrix.png")
plt.show()

#### Duplicate `id` / `imdb_id` check

In [ ]:
tmdb_dupe_ids = int(tmdb_raw["id"].duplicated().sum())
print(f"TMDB duplicate ids: {tmdb_dupe_ids}")

In [ ]:
# Do duplicate-id rows actually agree on core identity fields, or do they
# genuinely conflict (which would mean the same id was reused across two
# different movies, not just re-entered with more fields filled in)?
key_cols = ["imdb_id", "title", "release_date", "budget", "revenue", "vote_average"]
tmdb_dupe_id_rows = tmdb_raw[tmdb_raw["id"].duplicated(keep=False)]
n_distinct_non_null = tmdb_dupe_id_rows.groupby("id")[key_cols].agg(lambda s: s.dropna().nunique())
conflicting_ids = n_distinct_non_null[(n_distinct_non_null > 1).any(axis=1)]
n_dupe_id_groups = tmdb_dupe_id_rows["id"].nunique()
print(f"{len(conflicting_ids)} of {n_dupe_id_groups} duplicate-id groups "
      f"have conflicting (not just missing-vs-present) values in {key_cols}")

tmdb_dupe_id_rows[tmdb_dupe_id_rows["id"].isin(conflicting_ids.index)][["id"] + key_cols].sort_values("id")

#### `vote_average` / `vote_count` zero combinations

In [ ]:
vote_avg_zero = tmdb_raw["vote_average"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((vote_avg_zero & vote_count_zero).sum())
n_avg_nonzero_count_zero = int((~vote_avg_zero & vote_count_zero).sum())
n_avg_zero_count_nonzero = int((vote_avg_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~vote_avg_zero & ~vote_count_zero).sum())

print(f"vote_average==0 and vote_count==0:    {n_both_zero}")
print(f"vote_average!=0 and vote_count==0:    {n_avg_nonzero_count_zero}")
print(f"vote_average==0 and vote_count!=0:    {n_avg_zero_count_nonzero}")
print(f"vote_average!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                                {len(tmdb_raw)}")

# DECISION: vote_count==0 means zero votes were cast, so vote_average cannot
# be anything but 0 in that case -- an average requires at least one value
# to average. The 582 rows above with vote_count==0 but vote_average!=0 are
# therefore logically impossible, not just noisy: corrupted/erroneous records.
# TODO: remove (or recode vote_average to NaN) for these 582 rows as part of
# Section 1.2's cleaning, once the target-variable decision (deferred in
# Section 1.4) is finalized.

In [ ]:
# imdb_id is an external reference, not TMDB's own primary key, so unlike
# id above it can legitimately repeat if the same real movie was entered
# more than once in TMDB — a stronger duplicate-record signal than two rows
# merely sharing a title. "tt0000000" is IMDb's own placeholder for "no ID
# assigned" (not a real shared ID), so it's excluded here to avoid treating
# unrelated movies with an unknown imdb_id as false-positive duplicates.
tmdb_imdb_id_present = tmdb_raw["imdb_id"].dropna()
tmdb_imdb_id_present = tmdb_imdb_id_present[tmdb_imdb_id_present != "tt0000000"]
tmdb_dupe_imdb_id_mask = tmdb_imdb_id_present.duplicated(keep=False)
tmdb_dupe_imdb_ids = int(tmdb_imdb_id_present.duplicated().sum())
print(f"TMDB duplicate imdb_ids (excluding missing/placeholder values): {tmdb_dupe_imdb_ids} "
      f"out of {len(tmdb_imdb_id_present)} non-null, non-placeholder values")

tmdb_dupe_imdb_id_rows = tmdb_raw.loc[
    tmdb_imdb_id_present[tmdb_dupe_imdb_id_mask].index,
    ["id", "imdb_id", "title", "release_date"],
].sort_values("imdb_id")
tmdb_dupe_imdb_id_rows.head(20)

### 1.2 TMDB cleaning: status filter, deduplication & zero-coding

In [ ]:
tmdb = tmdb_raw.copy()

#### Recode `budget`/`revenue` zeros as NaN

In [ ]:
zero_recode_report = {}
for col in ["budget", "revenue"]:
    pct_zero = float((tmdb[col] == 0).mean() * 100)
    zero_recode_report[col] = pct_zero
    tmdb[col] = tmdb[col].replace(0, np.nan)
    print(f"{col}: {pct_zero:.1f}% of rows were 0 -> recoded as NaN")

#### Recode `runtime` zeros as NaN

In [ ]:
pct_zero_runtime = float((tmdb["runtime"] == 0).mean() * 100)
tmdb["runtime"] = tmdb["runtime"].replace(0, np.nan)
print(f"runtime: {pct_zero_runtime:.1f}% of rows were 0 -> recoded as NaN")

#### Recode placeholder `imdb_id` values as NaN

In [ ]:
tmdb_imdb_id_placeholder_pct = float((tmdb["imdb_id"] == "tt0000000").mean() * 100)
tmdb["imdb_id"] = tmdb["imdb_id"].replace("tt0000000", np.nan)
print(f"imdb_id: {tmdb_imdb_id_placeholder_pct:.3f}% of rows were the placeholder "
      f"'tt0000000' -> recoded as NaN")

#### Recode negative `runtime`/`revenue` values as NaN

In [ ]:
tmdb_neg_recode_report = {}
for col in ["runtime", "revenue"]:
    pct_negative = float((tmdb[col] < 0).mean() * 100)
    tmdb_neg_recode_report[col] = pct_negative
    tmdb[col] = tmdb[col].where(tmdb[col] >= 0, np.nan)
    print(f"{col}: {pct_negative:.4f}% of rows were negative -> recoded as NaN")

#### Missing values after recode

In [ ]:
tmdb_missing_recode = pd.DataFrame(
    {
        "n_missing": tmdb.isnull().sum(),
        "pct_missing": tmdb.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
tmdb_missing_recode

#### Filter to `status == 'Released'`

In [ ]:
status_counts = tmdb_raw["status"].value_counts(dropna=False)
status_pct = (status_counts / len(tmdb_raw) * 100).round(2)
pd.DataFrame({"count": status_counts, "pct": status_pct})

In [ ]:
# Verify the claim behind the filter across every column, not just the
# handful expected to be post-release-only.
released_mask = tmdb_raw["status"] == "Released"
missingness_by_status = pd.DataFrame({
    "% missing (Released)": tmdb_raw.loc[released_mask].isnull().mean() * 100,
    "% missing (non-Released)": tmdb_raw.loc[~released_mask].isnull().mean() * 100,
})
missingness_by_status["gap"] = missingness_by_status["% missing (non-Released)"] - missingness_by_status["% missing (Released)"]
missingness_by_status.round(2).sort_values("gap", ascending=False)

In [ ]:
shape_before_status_filter = tmdb.shape
tmdb = tmdb[tmdb["status"] == "Released"].copy()
tmdb_shape_after_status_filter = tmdb.shape
print(f"TMDB shape before status filter: {shape_before_status_filter}")
print(f"TMDB shape after status filter: {tmdb_shape_after_status_filter}")

#### Drop duplicate `id` rows (keep most complete)

In [ ]:
# DECISION POINT: "most complete" is currently just the row with the
# fewest total NaNs across ALL columns, treating every column as equally
# important. A row missing 2 minor fields (e.g. homepage, poster_path)
# scores the same as one missing 2 consequential ones (e.g. budget,
# revenue). Ties are broken arbitrarily by original row order (stable sort).
# TODO: revisit with column-weighted completeness (e.g. weight budget/
# revenue/overview higher than homepage/poster_path) if this heuristic
# turns out to pick the wrong row in practice.
tmdb_shape_before_dedup = tmdb.shape
tmdb = (
    tmdb.assign(_null_count=tmdb.isnull().sum(axis=1))
    .sort_values("_null_count")
    .drop_duplicates(subset="id", keep="first")
    .drop(columns="_null_count")
    .sort_index()
)
tmdb_shape_after_dedup = tmdb.shape
tmdb_n_dupe_ids_dropped = tmdb_shape_before_dedup[0] - tmdb_shape_after_dedup[0]
print(f"TMDB: dropped {tmdb_n_dupe_ids_dropped} duplicate-id rows (kept the most complete row per id)")
print(f"Shape before dedup: {tmdb_shape_before_dedup}, after: {tmdb_shape_after_dedup}")

### 1.3 Candidate outcome variable distributions

#### TMDB `vote_average` distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb["vote_average"].dropna().hist(bins=40, ax=ax)
ax.set_title("TMDB: vote_average distribution")
ax.set_xlabel("vote_average")
ax.set_ylabel("count")
savefig(fig, "00_tmdb_vote_average_hist.png")
plt.show()

In [ ]:
for threshold in [1, 5, 10, 20, 50]:
    print(f"vote_count >= {threshold}: {(tmdb['vote_count'] >= threshold).sum():,} rows")

In [ ]:
vote_avg_zero = tmdb_raw["vote_average"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((vote_avg_zero & vote_count_zero).sum())
n_avg_nonzero_count_zero = int((~vote_avg_zero & vote_count_zero).sum())
n_avg_zero_count_nonzero = int((vote_avg_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~vote_avg_zero & ~vote_count_zero).sum())

print(f"vote_average==0 and vote_count==0:    {n_both_zero}")
print(f"vote_average!=0 and vote_count==0:    {n_avg_nonzero_count_zero}")
print(f"vote_average==0 and vote_count!=0:    {n_avg_zero_count_nonzero}")
print(f"vote_average!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                                {len(tmdb_raw)}")

# DECISION: vote_count==0 means zero votes were cast, so vote_average cannot
# be anything but 0 in that case -- an average requires at least one value
# to average. The 582 rows above with vote_count==0 but vote_average!=0 are
# therefore logically impossible, not just noisy: corrupted/erroneous records.
# TODO: remove (or recode vote_average to NaN) for these 582 rows as part of
# Section 1.2's cleaning, once the target-variable decision (deferred in
# Section 1.4) is finalized.

## 2. Numerical features (Section 2a)

In [ ]:
TMDB_NUM_COLS = [c for c in ["vote_average", "vote_count", "revenue", "runtime", "budget", "popularity"] if c in tmdb.columns]
TMDB_TARGET = "vote_average"

print("TMDB numeric columns:", TMDB_NUM_COLS)

### Descriptive stats & outliers

#### `describe()` and physically impossible values

In [ ]:
tmdb[TMDB_NUM_COLS].describe()

In [ ]:
impossible_flags = []
if "runtime" in tmdb.columns:
    impossible_flags.append(("TMDB", "runtime == 0", int((tmdb["runtime"] == 0).sum())))
    impossible_flags.append(("TMDB", "runtime < 0", int((tmdb["runtime"] < 0).sum())))
if "revenue" in tmdb.columns:
    impossible_flags.append(("TMDB", "revenue < 0", int((tmdb["revenue"] < 0).sum())))
if "budget" in tmdb.columns:
    impossible_flags.append(("TMDB", "budget < 0", int((tmdb["budget"] < 0).sum())))

pd.DataFrame(impossible_flags, columns=["dataset", "check", "n_rows"])

In [ ]:
tmdb_neg_recode_report = {}
for col in ["runtime", "revenue"]:
    pct_negative = float((tmdb[col] < 0).mean() * 100)
    tmdb_neg_recode_report[col] = pct_negative
    tmdb[col] = tmdb[col].where(tmdb[col] >= 0, np.nan)
    print(f"{col}: {pct_negative:.4f}% of rows were negative -> recoded as NaN")

#### Outlier detection: IQR (1.5x) vs z-score (|z|>3)

In [ ]:
tmdb_outlier_summary = pd.DataFrame({
    "column": TMDB_NUM_COLS,
    "iqr_outliers": [iqr_outlier_count(tmdb[c]) for c in TMDB_NUM_COLS],
    "zscore_outliers": [zscore_outlier_count(tmdb[c]) for c in TMDB_NUM_COLS],
})
tmdb_outlier_summary["disagreement"] = (
    tmdb_outlier_summary["iqr_outliers"] - tmdb_outlier_summary["zscore_outliers"]
).abs()
tmdb_outlier_summary

IQR and z-score disagree substantially wherever a column is skewed rather
than roughly normal: IQR is robust to skew and flags points relative to the
central 50%, while z-score assumes a symmetric distribution and under- or
over-counts outliers accordingly. Treat the two counts as complementary,
not interchangeable.

#### Histogram

In [ ]:
fig, axes = plt.subplots(1, len(TMDB_NUM_COLS), figsize=(4 * len(TMDB_NUM_COLS), 5))
for ax, col in zip(np.atleast_1d(axes), TMDB_NUM_COLS):
    values = tmdb[col].dropna()
    values = values[values > 0]
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins)
    ax.set_xscale("log")
    ax.set_title(col)
fig.suptitle("TMDB: numeric feature histograms (log scale, positive values only)")
savefig(fig, "00_tmdb_histograms.png")
plt.show()

#### Boxplots

In [ ]:
fig, axes = plt.subplots(1, len(TMDB_NUM_COLS), figsize=(4 * len(TMDB_NUM_COLS), 5))
for ax, col in zip(np.atleast_1d(axes), TMDB_NUM_COLS):
    tmdb.boxplot(column=col, ax=ax)
    ax.set_title(col)
fig.suptitle("TMDB: numeric feature boxplots")
savefig(fig, "00_tmdb_boxplots.png")
plt.show()

### Skewness & transforms

#### Skewness

In [ ]:
# DECISION POINT: skewness flagging threshold — currently |skew| > 2, the
# more lenient end of common convention (some sources treat |skew| > 1 as
# already "highly skewed"; > 2 follows George & Mallery's more relaxed
# guideline). Columns with skew ~1.0-2.0 won't be flagged at this setting.
# TODO: revisit this threshold (e.g. try 1.0) if downstream modeling shows
# unflagged columns in that 1-2 range are still causing problems.
tmdb_skew = tmdb[TMDB_NUM_COLS].skew().sort_values(ascending=False)
tmdb_skew_flagged = tmdb_skew[tmdb_skew.abs() > 2]
print("TMDB skewness:")
print(tmdb_skew)
print()
print("Flagged (|skew| > 2) as log-transform / quantile-binning candidates:")
print(tmdb_skew_flagged)

### Correlations

#### Correlation matrix & multicollinearity flags

In [ ]:
tmdb_corr = tmdb[TMDB_NUM_COLS].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(tmdb_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("TMDB: numeric feature correlation matrix")
savefig(fig, "00_tmdb_correlation_heatmap.png")
plt.show()

In [ ]:
tmdb_high_corr_pairs = high_corr_pairs(tmdb_corr, threshold=0.75)
print("TMDB pairs with |r| > 0.75 (multicollinearity risk):")
tmdb_high_corr_pairs

In [ ]:
tmdb_target_corr = tmdb_corr[TMDB_TARGET].drop(TMDB_TARGET).abs().sort_values(ascending=False)
print(f"TMDB numeric features ranked by |r| with {TMDB_TARGET}:")
tmdb_target_corr

## 3. Categorical features (Section 2b)

In [ ]:
# "status" is excluded here: Section 1 already filtered tmdb to
# status == 'Released', so it is now a constant column post-cleaning and
# uninformative for cardinality/long-tail analysis.
TMDB_CAT_COLS = [c for c in ["original_language"] if c in tmdb.columns]
print("TMDB categorical columns:", TMDB_CAT_COLS)

### Cardinality & distributions

#### Cardinality

In [ ]:
cardinality_table = pd.DataFrame({
    "dataset": "TMDB", "column": TMDB_CAT_COLS,
    "cardinality": [tmdb[c].nunique() for c in TMDB_CAT_COLS],
})
cardinality_table

#### Value counts & long-tail categories

In [ ]:
for c in TMDB_CAT_COLS:
    vc = tmdb[c].value_counts(normalize=True)
    print(f"--- TMDB: {c} (top 5) ---")
    print(vc.head(5))
    long_tail = vc[vc < 0.01]
    print(f"{len(long_tail)} long-tail categories (<1% share each) — candidates for binning into 'Other'")
    print()

#### Long-tail visualization

In [ ]:
CAT_COLS_TO_PLOT = {
    "TMDB": TMDB_CAT_COLS,
}

for dataset_name, df_, cols in [("TMDB", tmdb, CAT_COLS_TO_PLOT["TMDB"])]:
    for c in cols:
        vc = df_[c].value_counts(normalize=True)
        top20 = vc.head(20)
        other_share = vc.iloc[20:].sum()
        plot_data = top20.copy()
        if other_share > 0:
            plot_data["Other"] = other_share

        fig, ax = plt.subplots(figsize=(10, 5))
        plot_data.plot(kind="bar", ax=ax)
        ax.set_title(f"{dataset_name}: {c} category frequency (top 20 + Other)")
        ax.set_ylabel("share of rows")
        ax.set_xlabel(c)
        plt.xticks(rotation=45, ha="right")
        fig.tight_layout()
        savefig(fig, f"00_{dataset_name.lower()}_{c}_long_tail.png")
        plt.show()

### Empty-string missingness (distinct from Section 1's `isnull()` check)

In [ ]:
empty_string_report(tmdb, TMDB_CAT_COLS)

### Binary-looking field audit

### Entity-embedding dimension heuristic

Suggested embedding dimension per categorical field, using
`min(50, (cardinality // 2) + 1)` (Guo & Berkhahn, 2016, *Entity Embeddings
of Categorical Variables*).

In [ ]:
def embedding_dim_table(df, cols, dataset_name):
    table = pd.DataFrame({"dataset": dataset_name, "column": cols})
    table["cardinality"] = [df[c].nunique() for c in cols]
    table["suggested_embedding_dim"] = table["cardinality"].apply(entity_embedding_dim)
    return table

embedding_dim_summary = embedding_dim_table(tmdb, TMDB_CAT_COLS, "TMDB")
embedding_dim_summary

### Fairness-relevant grouping

In [ ]:
FAIRNESS_MIN_GROUP_SIZE = 100

tmdb_lang_counts = tmdb["original_language"].value_counts()
tmdb_small_lang_groups = tmdb_lang_counts[tmdb_lang_counts < FAIRNESS_MIN_GROUP_SIZE]
print(f"TMDB original_language: {len(tmdb_small_lang_groups)} / {len(tmdb_lang_counts)} groups "
      f"have < {FAIRNESS_MIN_GROUP_SIZE} samples — unreliable for later fairness metrics:")
tmdb_small_lang_groups

## 4. Free-text string features (Section 2c)

Covers TMDB `overview` — the long free-text field in the dataset.

### Free-text field audit

Scans every string-typed column (not just `overview`/`tagline`) for its uniqueness ratio (`nunique / non-null count`), to check whether any other fields deserve the same free-text treatment. List-valued raw string columns (`genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`) are excluded here since they are handled separately in Section 5 -- their raw combined-string form isn't the right lens for cardinality.

A high ratio (close to 1.0) doesn't automatically mean a column needs the full text-quality treatment: identifier/URL fields (`imdb_id`, `homepage`, `poster_path`, `backdrop_path`) will also show high uniqueness but aren't natural language, so they're not worth token-length/language-detection/HTML checks. The genuine candidates to look for here are other short-to-medium narrative text fields, e.g. `title`/`original_title` -- both now covered below.

In [ ]:
LIST_VALUED_RAW_COLS = {
    "TMDB": ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"],
}

def high_cardinality_scan(df, dataset_name):
    exclude = LIST_VALUED_RAW_COLS.get(dataset_name, [])
    rows = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            non_null = df[c].notna().sum()
            nunique = df[c].nunique()
            ratio = nunique / non_null if non_null else 0.0
            rows.append((c, nunique, non_null, ratio))
    return pd.DataFrame(
        rows, columns=["column", "nunique", "non_null_count", "uniqueness_ratio"]
    ).sort_values("uniqueness_ratio", ascending=False)

tmdb_cardinality_scan = high_cardinality_scan(tmdb, "TMDB")
print("TMDB string columns by uniqueness ratio:")
tmdb_cardinality_scan

### TMDB

#### `overview`

##### Nulls & empty strings

In [ ]:
tmdb_overview_nulls, tmdb_overview_empty = text_field_report(tmdb, "overview", "TMDB overview")

##### Character length distribution

In [ ]:
tmdb_overview_len = tmdb["overview"].dropna().astype(str).str.len()
tmdb_overview_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_overview_len.hist(bins=50, ax=ax)
ax.set_title("TMDB overview: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "00_tmdb_overview_length_hist.png")
plt.show()

##### Token length (paraphrase-multilingual-mpnet-base-v2 tokenizer, 20000-row sample)

In [ ]:
tmdb_overview_token_lengths, tmdb_overview_pct_over_128 = token_length_stats(tmdb["overview"])
print(f"TMDB overview: {tmdb_overview_pct_over_128:.1f}% of sampled rows exceed 128 tokens")
tmdb_overview_token_lengths.describe()

##### Non-English sample flagging (5000-row sample)

In [ ]:
tmdb_overview_non_english, tmdb_overview_pct_non_english, tmdb_overview_pct_too_short, tmdb_overview_pct_uncertain = flag_non_english(tmdb["overview"])
tmdb_overview_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_overview_html_pct = html_contamination_pct(tmdb["overview"])
print(f"TMDB overview: {tmdb_overview_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_overview_dupes, tmdb_overview_n_dupe_rows = near_duplicate_report(tmdb, "overview", id_col="id")
tmdb_overview_dupes.head(20)

#### `tagline`

##### Nulls & empty strings

In [ ]:
tmdb_tagline_nulls, tmdb_tagline_empty = text_field_report(tmdb, "tagline", "TMDB tagline")

##### Character length distribution

In [ ]:
tmdb_tagline_len = tmdb["tagline"].dropna().astype(str).str.len()
tmdb_tagline_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_tagline_len.hist(bins=50, ax=ax)
ax.set_title("TMDB tagline: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "00_tmdb_tagline_length_hist.png")
plt.show()

##### Token length (paraphrase-multilingual-mpnet-base-v2 tokenizer, 20000-row sample)

In [ ]:
tmdb_tagline_token_lengths, tmdb_tagline_pct_over_128 = token_length_stats(tmdb["tagline"])
print(f"TMDB tagline: {tmdb_tagline_pct_over_128:.1f}% of sampled rows exceed 128 tokens")
tmdb_tagline_token_lengths.describe()

##### Non-English sample flagging (5000-row sample)

In [ ]:
tmdb_tagline_non_english, tmdb_tagline_pct_non_english, tmdb_tagline_pct_too_short, tmdb_tagline_pct_uncertain = flag_non_english(tmdb["tagline"])
tmdb_tagline_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_tagline_html_pct = html_contamination_pct(tmdb["tagline"])
print(f"TMDB tagline: {tmdb_tagline_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_tagline_dupes, tmdb_tagline_n_dupe_rows = near_duplicate_report(tmdb, "tagline", id_col="id")
tmdb_tagline_dupes.head(20)

#### `title`

##### Nulls & empty strings

In [ ]:
tmdb_title_nulls, tmdb_title_empty = text_field_report(tmdb, "title", "TMDB title")

##### Character length distribution

In [ ]:
tmdb_title_len = tmdb["title"].dropna().astype(str).str.len()
tmdb_title_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_title_len.hist(bins=50, ax=ax)
ax.set_title("TMDB title: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "00_tmdb_title_length_hist.png")
plt.show()

##### Token length (paraphrase-multilingual-mpnet-base-v2 tokenizer, 20000-row sample)

In [ ]:
tmdb_title_token_lengths, tmdb_title_pct_over_128 = token_length_stats(tmdb["title"])
print(f"TMDB title: {tmdb_title_pct_over_128:.1f}% of sampled rows exceed 128 tokens")
tmdb_title_token_lengths.describe()

##### Non-English sample flagging (5000-row sample)

In [ ]:
tmdb_title_non_english, tmdb_title_pct_non_english, tmdb_title_pct_too_short, tmdb_title_pct_uncertain = flag_non_english(tmdb["title"])
tmdb_title_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_title_html_pct = html_contamination_pct(tmdb["title"])
print(f"TMDB title: {tmdb_title_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_title_dupes, tmdb_title_n_dupe_rows = near_duplicate_report(tmdb, "title", id_col="id")
tmdb_title_dupes.head(20)

#### `original_title`

##### Nulls & empty strings

In [ ]:
tmdb_original_title_nulls, tmdb_original_title_empty = text_field_report(tmdb, "original_title", "TMDB original_title")

##### Character length distribution

In [ ]:
tmdb_original_title_len = tmdb["original_title"].dropna().astype(str).str.len()
tmdb_original_title_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_original_title_len.hist(bins=50, ax=ax)
ax.set_title("TMDB original_title: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "00_tmdb_original_title_length_hist.png")
plt.show()

##### Token length (paraphrase-multilingual-mpnet-base-v2 tokenizer, 20000-row sample)

In [ ]:
tmdb_original_title_token_lengths, tmdb_original_title_pct_over_128 = token_length_stats(tmdb["original_title"])
print(f"TMDB original_title: {tmdb_original_title_pct_over_128:.1f}% of sampled rows exceed 128 tokens")
tmdb_original_title_token_lengths.describe()

##### Non-English sample flagging (5000-row sample)

In [ ]:
tmdb_original_title_non_english, tmdb_original_title_pct_non_english, tmdb_original_title_pct_too_short, tmdb_original_title_pct_uncertain = flag_non_english(tmdb["original_title"])
tmdb_original_title_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_original_title_html_pct = html_contamination_pct(tmdb["original_title"])
print(f"TMDB original_title: {tmdb_original_title_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_original_title_dupes, tmdb_original_title_n_dupe_rows = near_duplicate_report(tmdb, "original_title", id_col="id")
tmdb_original_title_dupes.head(20)

## 5. List-valued features (Section 2d)

Covers TMDB `genres`, `keywords`, `production_companies`,
`production_countries`, and `spoken_languages`. TMDB stores these as
comma-separated plain strings rather than JSON, so `parse_list_field()`
(defined in Setup) tries `json.loads` / `ast.literal_eval` first and
falls back to a comma split otherwise.

### Parsing verification

In [ ]:
for col, df_ in [
    ("genres", tmdb),
    ("keywords", tmdb),
    ("production_companies", tmdb),
    ("production_countries", tmdb),
    ("spoken_languages", tmdb),
]:
    sample_val = df_[col].dropna().iloc[0]
    print(f"--- {col} ---")
    print("raw:", sample_val)
    print("parsed:", parse_list_field(sample_val))
    print()

In [ ]:
tmdb["genres_list"] = tmdb["genres"].apply(parse_list_field)
tmdb["keywords_list"] = tmdb["keywords"].apply(parse_list_field)
tmdb["production_companies_list"] = tmdb["production_companies"].apply(parse_list_field)
tmdb["production_countries_list"] = tmdb["production_countries"].apply(parse_list_field)
tmdb["spoken_languages_list"] = tmdb["spoken_languages"].apply(parse_list_field)

### TMDB `genres`

#### Vocabulary & frequency

In [ ]:
tmdb_genres_summary = list_field_summary(tmdb, "genres_list")
tmdb_genres_vocab = tmdb_genres_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_genres_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_genres_summary['long_tail_items']}")
tmdb_genres_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_genres_summary["length_describe"])
print(f"% rows with empty list: {tmdb_genres_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_genres_vocab, len(tmdb))

#### Co-occurrence of top 20 genres

In [ ]:
top_item_cooccurrence(tmdb["genres_list"], top_n=20)

### TMDB `keywords`

#### Vocabulary & frequency

In [ ]:
tmdb_keywords_summary = list_field_summary(tmdb, "keywords_list")
tmdb_keywords_vocab = tmdb_keywords_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_keywords_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_keywords_summary['long_tail_items']}")
tmdb_keywords_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
# DECISION POINT: ~75% of rows have an empty keywords list. "Empty" here
# doesn't yet distinguish genuinely-missing (NaN, never tagged) from
# explicitly-empty (parsed to [] from an empty string) — same disguised-
# missingness risk seen elsewhere in this dataset.
# TODO: check whether emptiness correlates with release_date/popularity
# (structured vs. random), and decide how encoding should represent "no
# keywords" before treating this field as a primary feature.
print(tmdb_keywords_summary["length_describe"])
print(f"% rows with empty list: {tmdb_keywords_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_keywords_vocab, len(tmdb))

#### Co-occurrence of top 20 keywords

In [ ]:
top_item_cooccurrence(tmdb["keywords_list"], top_n=20)

### TMDB `production_companies`

#### Vocabulary & frequency

In [ ]:
tmdb_production_companies_summary = list_field_summary(tmdb, "production_companies_list")
tmdb_production_companies_vocab = tmdb_production_companies_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_production_companies_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_production_companies_summary['long_tail_items']}")
tmdb_production_companies_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_production_companies_summary["length_describe"])
print(f"% rows with empty list: {tmdb_production_companies_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_production_companies_vocab, len(tmdb))

#### Co-occurrence of top 20 production companies

In [ ]:
top_item_cooccurrence(tmdb["production_companies_list"], top_n=20)

### TMDB `production_countries`

#### Vocabulary & frequency

In [ ]:
tmdb_production_countries_summary = list_field_summary(tmdb, "production_countries_list")
tmdb_production_countries_vocab = tmdb_production_countries_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_production_countries_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_production_countries_summary['long_tail_items']}")
tmdb_production_countries_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_production_countries_summary["length_describe"])
print(f"% rows with empty list: {tmdb_production_countries_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_production_countries_vocab, len(tmdb))

#### Co-occurrence of top 20 production countries

In [ ]:
top_item_cooccurrence(tmdb["production_countries_list"], top_n=20)

### TMDB `spoken_languages`

#### Vocabulary & frequency

In [ ]:
tmdb_spoken_languages_summary = list_field_summary(tmdb, "spoken_languages_list")
tmdb_spoken_languages_vocab = tmdb_spoken_languages_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_spoken_languages_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_spoken_languages_summary['long_tail_items']}")
tmdb_spoken_languages_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_spoken_languages_summary["length_describe"])
print(f"% rows with empty list: {tmdb_spoken_languages_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_spoken_languages_vocab, len(tmdb))

#### Co-occurrence of top 20 spoken languages

In [ ]:
top_item_cooccurrence(tmdb["spoken_languages_list"], top_n=20)

## Log summary

In [ ]:
findings = {
    "TMDB rows (raw -> released, zero-coded)": f"{tmdb_shape_raw[0]} -> {tmdb.shape[0]}",
    "TMDB memory footprint (deep)": f"{tmdb_mem_mb:,.1f} MB",
    "TMDB most predictive numeric features (|r| with vote_average)": tmdb_target_corr.head(3).round(3).to_dict(),
    "Entity-embedding dimension table (see notebook Section 3)": embedding_dim_summary.set_index(["dataset", "column"])["suggested_embedding_dim"].to_dict(),
    "TMDB overview: % > 128 tokens (sampled)": f"{tmdb_overview_pct_over_128:.1f}%",
    "TMDB overview: non-English / too short / uncertain (sampled)": (
        f"{tmdb_overview_pct_non_english:.1f}% / {tmdb_overview_pct_too_short:.1f}% / "
        f"{tmdb_overview_pct_uncertain:.1f}%"
    ),
    "TMDB overview: % HTML-contaminated": f"{tmdb_overview_html_pct:.2f}%",
    "TMDB genres vocabulary size": tmdb_genres_vocab,
    "TMDB keywords vocabulary size": tmdb_keywords_vocab,
    "Multi-hot vs pooled-embedding recommendation": (
        "genres: multi-hot feasible; keywords: use pooled "
        "SBERT-of-item-names if vocab_size > 5000 (see Section 5 warnings above)"
    ),
}

log_findings(
    title="00 Characterization: TMDB — summary",
    findings=findings,
    figures=[
        "reports/figures/00_tmdb_vote_average_hist.png",
        "reports/figures/00_tmdb_boxplots.png",
        "reports/figures/00_tmdb_correlation_heatmap.png",
        "reports/figures/00_tmdb_overview_length_hist.png",
    ],
)
print("Logged summary to reports/eda_log.md")